This notebook demonstrates how to use the clemcore Pettingzoo integration to run a
single-player game with the game **"Wordle"**, and how to plug in a self-hosted
OpenAI-compatible model server as an agent backend.

You will learn how to:

1. Set up the environment and install dependencies.
2. Register and configure a model as an agent backend.
3. Implement a simple custom guesser agent.
4. Run an automated gameplay session and inspect the interactions.

This example is similar to the Gymnasium-style [Wordle example](https://github.com/github/phisad/playpen/blob/main/examples/gymnasium/wordle.ipynb).
However, the loop over the environment is different.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/phisad/playpen/blob/main/examples/pettingzoo/wordle.ipynb
)

# 1. Environment setup and dependency installation

## 1.1. Setup environment

We start by specifying:

- The game name (here we use `"wordle"`, but this pattern works for any 1‑player game).
- `CLEMBENCH_HOME`, the local path where the `clembench` repository is located.
  This environment variable is used by the CLI and Python APIs to locate games.

In [1]:
import os

# Specify the game name here (this code can be adapted to any 1-player game)
GAME_NAME = "wordle"

# Local clone location of the clembench repository
CLEMBENCH_HOME = os.path.expanduser("~/git/clembench")

# Expose CLEMBENCH_HOME so the clem framework can find the games
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

## 1.2. Install games and dependencies

In this step we:

1. Clone the `clembench` repository (skip if you already have it).
2. Install its Python dependencies into the current Jupyter kernel.
3. Verify that `clembench` is installed and that the game is visible.

You should:

- See a valid version printed by `clem --version`.
- See `wordle` listed by `clem list games -s wordle`.

If you do **not** see `wordle`, double‑check `CLEMBENCH_HOME` and that the clone succeeded.

In [ ]:
# Clone the clembench repo (safe to re-run; git will warn if it already exists)
!git clone https://github.com/clp-research/clembench $CLEMBENCH_HOME

# Install the requirements into the Python kernel
%pip install -r $CLEMBENCH_HOME/requirements.txt

# Make tqdm usable in Jupyter notebooks
%pip install --upgrade ipywidgets jupyter_client

In [15]:
# Sanity check: version + confirm that the game is an available game
!clem --version
!clem list games -s $GAME_NAME

clem 3.3.5
Listing all available games (use -v option to see the whole specs)
Found '1' game specs that match the game_selector='{'game_name': 'taboo'}'
taboo:
 	Taboo game between two agents where one has to describe a word for
	the other to guess.


# 2. Register and configure a model backend

We will use a self-hosted OpenAI-compatible model server and register the model
under the name `"clp-chat"`.

This name (`"clp-chat"`) will later be used when we assign the model to a player
in the PettingZoo-style environment.

In [2]:
# Register the remote model under the name "clp-chat"
!clem register model -n "clp-chat" -v "backend=openai_compatible,model_id=Qwen/Qwen3-VL-30B-A3B-Instruct-FP8" --cwd

Updated model registry at /Users/philippsadler/Opts/Git/playpen/examples/pettingzoo/model_registry.json successfully: {"model_name":"clp-chat","backend":"openai_compatible","lookup_source":"/Users/philippsadler/Opts/Git/playpen/examples/pettingzoo/model_registry.json","model_id":"Qwen/Qwen3-VL-30B-A3B-Thinking-FP8"}


You can achieve the same registration programmatically if you prefer not to use
the CLI:

In [3]:
from clemcore.backends import ModelRegistry

registry = ModelRegistry.register("clp-chat", backend="openai_compatible", model_id="Qwen/Qwen3-VL-30B-A3B-Instruct-FP8")
registry.get_first_model_spec_that_unify_with("clp-chat")

ModelSpec({'model_name': 'clp-chat', 'backend': 'openai_compatible', 'lookup_source': '/Users/philippsadler/Opts/Git/playpen/examples/pettingzoo/model_registry.json', 'model_id': 'Qwen/Qwen3-VL-30B-A3B-Thinking-FP8'})

## 2.2. Register API key and base URL

To connect to your OpenAI-compatible model server, we also register:

- An API key
- An organisation (if applicable)
- A base URL for the endpoint

Replace the placeholders with your actual values.

In [4]:
API_KEY = "<insert-api-key-here>"
ORGANIZATION = "<insert-organisation-here>"
BASE_URL = "<insert-base-url-here>"

In [ ]:
# Register the credentials and base URL for the "openai_compatible" backend
!clem register key -n "openai_compatible" -v "api_key=$API_KEY,organisation=$ORGANIZATION,base_url=$BASE_URL" --cwd

Programmatic equivalent:

In [6]:
from clemcore.backends import KeyRegistry

KeyRegistry.register("openai_compatible", api_key=API_KEY, organisation=ORGANIZATION, base_url=BASE_URL, force_cwd=True)

KeyRegistry(file='/Users/philippsadler/Opts/Git/playpen/examples/pettingzoo/key.json', backends=[openai_compatible])

# 3. Implement a custom guesser agent

Next, we implement a simple custom guesser agent by subclassing `ClemAgent`.
The agent:

- Uses the `"clp-chat"` model backend.
- Generates a guess based on the full interaction history in the Wordle game.

This is just a toy example to show how you can plug in your own logic.

In [9]:
from playpen.agents import ClemAgent, ClemObservation
from clemcore.backends import load_model


class MyAgenticGuesser(ClemAgent):
    """
    Simple example agent for Wordle.

    It calls the "clp-chat" model with the current interaction history and
    uses the model's response as the next guess.
    """

    def __init__(self):
        super().__init__()
        self.model = load_model("clp-chat", gen_args=dict(temperature=0.7, max_tokens=None))

    def act(self, last: ClemObservation) -> str:
        # Use the full history (which usually already includes the last observation)
        _, _, response_text = self.model.generate_response(self.history)
        # Observe our own response in the interaction history
        self.observe(dict(role="user", content=response_text))
        return response_text


guesser = MyAgenticGuesser()

2026-01-09 13:41:05,177 - clemcore.backends - INFO - Found registered model spec that unifies with {"model_name":"clp-chat"} -> {'model_name': 'clp-chat', 'backend': 'openai_compatible', 'lookup_source': '/Users/philippsadler/Opts/Git/playpen/examples/pettingzoo/model_registry.json', 'model_id': 'Qwen/Qwen3-VL-30B-A3B-Thinking-FP8'}
2026-01-09 13:41:05,179 - clemcore.backends - INFO - Found registry entry for backend openai_compatible -> {'backend': 'openai_compatible', 'file_name': 'openai_compatible_api.py', 'file_path': '/Users/philippsadler/Opts/Git/playpen/venv/lib/python3.10/site-packages/clemcore/backends/openai_compatible_api.py', 'lookup_source': 'packaged'}
2026-01-09 13:41:05,180 - clemcore.backends - INFO - Dynamically import backend openai_compatible
2026-01-09 13:41:05,184 - clemcore.backends - INFO - Successfully loaded clp-chat model
2026-01-09 13:41:05,185 - clemcore.backends - INFO - Loading models took: 0:00:00.004892


# 4. Run an automated gameplay session

We now:

1. Create a PettingZoo-style environment for the Wordle game.
2. Use our custom guesser as `player_0`.
3. Run through a single episode step-by-step using the PettingZoo API.
4. Automatically run through the whole episode and log the interaction at each step.

The environment follows the standard PettingZoo interaction pattern:

- `env.agent_iter()` yields one `agent_id` at a time.
- `env.last()` returns the `(observation, reward, termination, truncation, info)`
  for the **current** agent.
- `env.step(action)` submits an action for the current agent and advances the game.

`termination` becomes `True` when the underlying game episode is logically over.
`truncation` becomes `True` if the episode ends early due to a time limit or
other external cutoff.

Setting `single_pass=False` (the default) allows iterating through all available
game instances multiple times. You can restrict which instances are used via the
`game_instance_filter` argument of the `env()` function.

In [5]:
from clemcore.clemgame import env, episode_results_folder_callbacks

# Create callbacks to record the interactions in a folder; here we name the folder after the models the agent uses
callbacks = episode_results_folder_callbacks(run_dir="clp-chat", result_dir_path="playpen-records", player_model_infos="MyAgenticDescriber")

game_env = env(
    "wordle",
    instances_filter=lambda sample: sample["game_instance"].get("game_id") in (1, 5, 10),
    single_pass=False,
    callbacks=callbacks
)
game_env.reset()

2026-01-09 13:21:17,559 - clemcore.cli - INFO - Found '1' game matching the game_selector="wordle"
2026-01-09 13:21:17,560 - clemcore.cli - INFO - {
  "game_name": "wordle",
  "description": "Wordle 5-letter word guessing game.",
  "main_game": "wordle",
  "players": 1,
  "image": "none",
  "languages": [
    "en"
  ],
  "benchmark": [
    "2.0"
  ],
  "regression": "small",
  "roles": [
    "Guesser"
  ],
  "game_path": "/Users/philippsadler/git/clembench/wordle"
}
2026-01-09 13:21:17,561 - clemcore.run - INFO - Loading game benchmark for wordle
2026-01-09 13:21:17,573 - clemcore.run - INFO - Loading game benchmark for wordle took: 0:00:00.010811
2026-01-09 13:21:17,580 - clemcore.run - INFO - Sub-select for wordle experiment high_frequency_words_no_clue_no_critic instances with game_ids: [1, 5, 10]
2026-01-09 13:21:17,581 - clemcore.run - INFO - Sub-select for wordle experiment medium_frequency_words_no_clue_no_critic instances with game_ids: [1, 5, 10]
2026-01-09 13:21:17,582 - clem

In [10]:
# Let's peek at the possible player ids (only available after reset, because reset() initiates the GameMaster)
print("possible agents:", game_env.possible_agents)
# In most cases, roles will be the content description of what player_0 and player_1 are
print("likely mapping:", game_env.unwrapped.game_master.game_spec["roles"])
agent_mapping = {"player_0": guesser}

possible agents: ['player_0']
likely mapping: ['Guesser']


In [7]:
# Now we can iterate through the agent turns, one step at a time (calling this cell multiple times)

try:
    agent_id = next(iter(game_env.agent_iter()))
    print("Agent_id:", agent_id)
    context, reward, termination, truncation, info = game_env.last()
    print("Reward:", reward, "Termination:", termination, "Truncation:", truncation, "Info:", info)
    if termination or truncation:
        response = None  # we step one more time to remove the agent from the env (final reward was observed in last)
    else:
        response = agent_mapping[agent_id](context)
    print("Context:", context)
    print("Response:", response)
    game_env.step(response)
except StopIteration:
    print("Episode finished.")
    guesser.reset()

Agent_id: player_0
Reward: 0.0 Termination: False Truncation: False Info: {}
Context: {'role': 'user', 'content': 'You are a language wizard who likes to guess words by using the given rules.\n\nWelcome to Wordle! You have six attempts to guess the target word, a valid English word of five lowercase letters (a-z). Please use the tags "explanation:" and "guess:" to provide a concise explanation for each guess.\n\nFor instance, if your guess is "apple", your response should be\nexplanation: this is a common five-letter English word, and I am starting my guess with this word.\nguess: apple\n\nAfter each guess, your answer will be validated, and you will receive feedback indicating which letters are correct (green), which letters are correct but in the wrong position (yellow), and which letters are incorrect (red). This feedback can be useful in determining which letters to include or exclude in your next guess.\n\nFor example, the feedback for "apple" might be:\nguess_feedback: a<yellow> 

In [11]:
# Now we can do everything automated as well
game_env.reset()
context_response_pairs = []
for agent_id in game_env.agent_iter():
    context, reward, termination, truncation, info = game_env.last()
    if termination or truncation:
        response = None  # we step one more time to remove the agent from the env (final reward was observed in last)
    else:
        response = agent_mapping[agent_id](context)
    context_response_pairs.append((context, response, reward))
    game_env.step(response)

guesser.reset()

print(f"Episode took these {len(context_response_pairs)} steps:")
print("-" * 20)
for idx, (context, response, reward) in enumerate(context_response_pairs):
    print(f"Step {idx} / Reward {reward:.2f}:")
    print(f"Guesser <- Context:", context)
    print(f"Guesser -> Response:", response)
    print("-" * 20)


Episode took these 2 steps:
--------------------
Step 0 / Reward 0.00:
Guesser <- Context: {'role': 'user', 'content': 'You are a language wizard who likes to guess words by using the given rules.\n\nWelcome to Wordle! You have six attempts to guess the target word, a valid English word of five lowercase letters (a-z). Please use the tags "explanation:" and "guess:" to provide a concise explanation for each guess.\n\nFor instance, if your guess is "apple", your response should be\nexplanation: this is a common five-letter English word, and I am starting my guess with this word.\nguess: apple\n\nAfter each guess, your answer will be validated, and you will receive feedback indicating which letters are correct (green), which letters are correct but in the wrong position (yellow), and which letters are incorrect (red). This feedback can be useful in determining which letters to include or exclude in your next guess.\n\nFor example, the feedback for "apple" might be:\nguess_feedback: a<yel